setup and the merge function

This defines merge_into_silver(df, name, keys). It matches incoming rows to silver on the business keys, updates rows that exist, inserts rows that don’t, and prints how many were inserted and updated.

In [0]:
from pyspark.sql import functions as F

catalog = "workspace"
schema = "de_ecommerce"
meta = ["source_file", "ingested_at"]
tables = ["orders", "order_items", "order_payments", "order_reviews"]

def bronze(name):
    return spark.table(f"{catalog}.{schema}.bronze_{name}")

def merge_into_silver(df, name, keys):
    df.createOrReplaceTempView(f"src_{name}")
    cond = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    res = spark.sql(f"""
        MERGE INTO {catalog}.{schema}.silver_{name} AS t
        USING src_{name} AS s
        ON {cond}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """).first()
    total = spark.table(f"{catalog}.{schema}.silver_{name}").count()
    print(f"{name}: inserted {res['num_inserted_rows']}, updated {res['num_updated_rows']}, table total {total}")

reset the silver order tables (run once)

This empties the 4 silver order tables so they can be rebuilt from the new bronze data.

In [0]:
for t in tables:
    spark.sql(f"TRUNCATE TABLE {catalog}.{schema}.silver_{t}")
    print(t, spark.table(f"{catalog}.{schema}.silver_{t}").count())

orders 0
order_items 0
order_payments 0
order_reviews 0


orders

This cleans the bronze orders with the Step 4 rules and merges them on order_id.

In [0]:
ts_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
           "order_delivered_customer_date", "order_estimated_delivery_date"]

df = bronze("orders").drop(*meta)
for c in ts_cols:
    df = df.withColumn(c, F.col(c).cast("timestamp"))
df = (df.dropDuplicates(["order_id"])
        .filter(F.col("order_id").isNotNull() & F.col("customer_id").isNotNull())
        .withColumn("order_status", F.lower(F.trim("order_status")))
        .filter(F.col("order_delivered_customer_date").isNull() |
                (F.col("order_delivered_customer_date") >= F.col("order_purchase_timestamp"))))
merge_into_silver(df, "orders", ["order_id"])

orders: inserted 0, updated 66638, table total 66638


items, payments, reviews

This does the same for the other three tables, with the Step 4 cleaning rules.

In [0]:
# ---- order_items ----
df = bronze("order_items").drop(*meta)
df = (df.dropDuplicates(["order_id", "order_item_id"])
        .filter(F.col("order_id").isNotNull() & F.col("product_id").isNotNull() & F.col("seller_id").isNotNull())
        .filter((F.col("price") > 0) & (F.col("freight_value") >= 0))
        .withColumn("shipping_limit_date", F.col("shipping_limit_date").cast("timestamp")))
merge_into_silver(df, "order_items", ["order_id", "order_item_id"])

# ---- order_payments ----
df = bronze("order_payments").drop(*meta)
df = (df.dropDuplicates(["order_id", "payment_sequential"])
        .filter(F.col("order_id").isNotNull())
        .filter(F.col("payment_value") >= 0)
        .filter(F.col("payment_type") != "not_defined"))
merge_into_silver(df, "order_payments", ["order_id", "payment_sequential"])

# ---- order_reviews ----
df = bronze("order_reviews").drop(*meta)
df = (df.withColumn("review_score", F.col("review_score").cast("int"))
        .withColumn("review_creation_date", F.col("review_creation_date").cast("timestamp"))
        .withColumn("review_answer_timestamp", F.col("review_answer_timestamp").cast("timestamp"))
        .filter(F.col("order_id").isNotNull() & F.col("review_score").between(1, 5))
        .dropDuplicates(["review_id", "order_id"]))
merge_into_silver(df, "order_reviews", ["review_id", "order_id"])

order_items: inserted 0, updated 75331, table total 75331
order_payments: inserted 0, updated 69898, table total 69898
order_reviews: inserted 0, updated 66560, table total 66560


check for duplicate keys

This counts keys that appear more than once in each silver table. It should be 0 everywhere.

In [0]:
keys = {"orders": ["order_id"],
        "order_items": ["order_id", "order_item_id"],
        "order_payments": ["order_id", "payment_sequential"],
        "order_reviews": ["review_id", "order_id"]}

for t, k in keys.items():
    tbl = spark.table(f"{catalog}.{schema}.silver_{t}")
    dups = tbl.groupBy(*k).count().filter("count > 1").count()
    print(f"{t}: total {tbl.count()} | duplicate keys: {dups}")

orders: total 66638 | duplicate keys: 0
order_items: total 75331 | duplicate keys: 0
order_payments: total 69898 | duplicate keys: 0
order_reviews: total 66560 | duplicate keys: 0
